# SQL in Python: Local Restaurant Foods Database

This notebook demonstrates how to use **SQL inside a Python notebook** with the built-in `sqlite3` module.

We'll build a small database about food served at local restaurants, made up of three related tables:

1. **`restaurants`** — basic info about each restaurant (name, cuisine, city, rating)
2. **`menu_items`** — dishes each restaurant serves (linked to `restaurants`)
3. **`orders`** — customer orders (linked to `menu_items`)

Then we'll run basic SQL commands: `SELECT`, `WHERE`, `DISTINCT`, `JOIN`, `GROUP BY`, `MAX`, `AVG`, `COUNT`, and `ORDER BY`.

`sqlite3` ships with Python, so no extra installation is needed. We'll also use `pandas` to display query results as nice tables.

## 1. Setup: connect to a SQLite database

We'll create the database as a file called `restaurant_foods.db` in the current directory. You can also use `':memory:'` for a temporary in-memory database.

In [1]:
import sqlite3
import pandas as pd

# Connect to (or create) the database file
conn = sqlite3.connect("restaurant_foods.db")
cursor = conn.cursor()

# Small helper to run a query and return a pandas DataFrame
def run_query(sql, params=None):
    return pd.read_sql_query(sql, conn, params=params)

print("Connected to restaurant_foods.db")

Connected to restaurant_foods.db


## 2. Create the `restaurants` table

Each row is one local restaurant.

In [2]:
cursor.execute("DROP TABLE IF EXISTS restaurants")

cursor.execute('''
CREATE TABLE restaurants (
    restaurant_id INTEGER PRIMARY KEY,
    name          TEXT NOT NULL,
    cuisine_type  TEXT NOT NULL,
    city          TEXT NOT NULL,
    rating        REAL,
    price_range   TEXT  -- $, $$, $$$
)
''')

restaurants_data = [
    (1, "Golden Dragon",       "Chinese",  "Springfield", 4.5, "$$"),
    (2, "Bella Pasta",         "Italian",  "Springfield", 4.2, "$$"),
    (3, "Taco Fiesta",         "Mexican",  "Shelbyville", 4.0, "$"),
    (4, "Sakura Sushi",        "Japanese", "Springfield", 4.7, "$$$"),
    (5, "The Burger Joint",    "American", "Shelbyville", 3.9, "$"),
    (6, "Spice Route",         "Indian",   "Capital City", 4.6, "$$"),
    (7, "Napoli Pizzeria",     "Italian",  "Capital City", 4.3, "$$"),
]

cursor.executemany(
    "INSERT INTO restaurants VALUES (?, ?, ?, ?, ?, ?)",
    restaurants_data
)
conn.commit()

run_query("SELECT * FROM restaurants")

,restaurant_id,name,cuisine_type,city,rating,price_range
0,1,Golden Dragon,Chinese,Springfield,4.5,$$
1,2,Bella Pasta,Italian,Springfield,4.2,$$
2,3,Taco Fiesta,Mexican,Shelbyville,4.0,$
3,4,Sakura Sushi,Japanese,Springfield,4.7,$$$
4,5,The Burger Joint,American,Shelbyville,3.9,$
5,6,Spice Route,Indian,Capital City,4.6,$$
6,7,Napoli Pizzeria,Italian,Capital City,4.3,$$


## 3. Create the `menu_items` table

Each row is a dish served at a specific restaurant (linked via `restaurant_id`).

In [3]:
cursor.execute("DROP TABLE IF EXISTS menu_items")

cursor.execute('''
CREATE TABLE menu_items (
    item_id       INTEGER PRIMARY KEY,
    restaurant_id INTEGER NOT NULL,
    item_name     TEXT NOT NULL,
    category      TEXT NOT NULL,  -- Appetizer, Entree, Dessert, Beverage
    price         REAL NOT NULL,
    is_vegetarian INTEGER NOT NULL,  -- 0 = no, 1 = yes
    FOREIGN KEY (restaurant_id) REFERENCES restaurants (restaurant_id)
)
''')

menu_items_data = [
    (1,  1, "Kung Pao Chicken",     "Entree",    14.99, 0),
    (2,  1, "Vegetable Spring Rolls","Appetizer", 6.99, 1),
    (3,  1, "Fried Rice",           "Entree",     9.99, 1),
    (4,  2, "Spaghetti Carbonara",  "Entree",    15.50, 0),
    (5,  2, "Margherita Pizza",     "Entree",    12.00, 1),
    (6,  2, "Tiramisu",             "Dessert",    7.50, 1),
    (7,  3, "Chicken Tacos (3)",    "Entree",     8.99, 0),
    (8,  3, "Guacamole & Chips",    "Appetizer",  5.99, 1),
    (9,  3, "Churros",              "Dessert",    4.50, 1),
    (10, 4, "Salmon Nigiri (2pc)",  "Appetizer", 6.00, 0),
    (11, 4, "Dragon Roll",          "Entree",    16.00, 0),
    (12, 4, "Miso Soup",            "Appetizer",  3.50, 1),
    (13, 5, "Classic Cheeseburger", "Entree",    10.99, 0),
    (14, 5, "Sweet Potato Fries",   "Appetizer",  4.99, 1),
    (15, 5, "Chocolate Milkshake",  "Beverage",   5.50, 1),
    (16, 6, "Chicken Tikka Masala", "Entree",    13.99, 0),
    (17, 6, "Vegetable Samosas",    "Appetizer",  5.50, 1),
    (18, 6, "Mango Lassi",          "Beverage",   3.99, 1),
    (19, 7, "Pepperoni Pizza",      "Entree",    13.50, 0),
    (20, 7, "Caprese Salad",        "Appetizer",  7.00, 1),
]

cursor.executemany(
    "INSERT INTO menu_items VALUES (?, ?, ?, ?, ?, ?)",
    menu_items_data
)
conn.commit()

run_query("SELECT * FROM menu_items LIMIT 5")

,item_id,restaurant_id,item_name,category,price,is_vegetarian
0,1,1,Kung Pao Chicken,Entree,14.99,0
1,2,1,Vegetable Spring Rolls,Appetizer,6.99,1
2,3,1,Fried Rice,Entree,9.99,1
3,4,2,Spaghetti Carbonara,Entree,15.50,0
4,5,2,Margherita Pizza,Entree,12.00,1


## 4. Create the `orders` table

Each row is a customer order for one menu item (linked via `item_id`).

In [4]:
cursor.execute("DROP TABLE IF EXISTS orders")

cursor.execute('''
CREATE TABLE orders (
    order_id      INTEGER PRIMARY KEY,
    item_id       INTEGER NOT NULL,
    customer_name TEXT NOT NULL,
    order_date    TEXT NOT NULL,   -- YYYY-MM-DD
    quantity      INTEGER NOT NULL,
    FOREIGN KEY (item_id) REFERENCES menu_items (item_id)
)
''')

orders_data = [
    (1,  1,  "Alice Chen",    "2026-08-01", 2),
    (2,  3,  "Alice Chen",    "2026-08-01", 1),
    (3,  5,  "Brian Kim",     "2026-08-02", 1),
    (4,  6,  "Brian Kim",     "2026-08-02", 2),
    (5,  7,  "Carla Diaz",    "2026-08-02", 3),
    (6,  9,  "Carla Diaz",    "2026-08-02", 2),
    (7,  11, "Dan Osei",      "2026-08-03", 1),
    (8,  10, "Dan Osei",      "2026-08-03", 2),
    (9,  13, "Ella Novak",    "2026-08-03", 2),
    (10, 15, "Ella Novak",    "2026-08-03", 2),
    (11, 16, "Frank Ito",     "2026-08-04", 1),
    (12, 18, "Frank Ito",     "2026-08-04", 1),
    (13, 19, "Grace Lin",     "2026-08-04", 2),
    (14, 20, "Grace Lin",     "2026-08-04", 1),
    (15, 1,  "Henry Ford",    "2026-08-05", 1),
    (16, 4,  "Henry Ford",    "2026-08-05", 1),
    (17, 5,  "Isla Brown",    "2026-08-05", 3),
    (18, 13, "Isla Brown",    "2026-08-06", 1),
    (19, 16, "Jack Wu",       "2026-08-06", 2),
    (20, 19, "Jack Wu",       "2026-08-06", 2),
    (21, 7,  "Alice Chen",    "2026-08-07", 1),
    (22, 11, "Brian Kim",     "2026-08-07", 1),
]

cursor.executemany(
    "INSERT INTO orders VALUES (?, ?, ?, ?, ?)",
    orders_data
)
conn.commit()

run_query("SELECT * FROM orders LIMIT 5")

,order_id,item_id,customer_name,order_date,quantity
0,1,1,Alice Chen,2026-08-01,2
1,2,3,Alice Chen,2026-08-01,1
2,3,5,Brian Kim,2026-08-02,1
3,4,6,Brian Kim,2026-08-02,2
4,5,7,Carla Diaz,2026-08-02,3


## 5. Basic SQL commands

Now that all three tables are populated, let's run through some fundamental SQL operations.

### 5.1 `SELECT` with `WHERE` — filter restaurants by rating

In [5]:
run_query('''
SELECT name, cuisine_type, city, rating
FROM restaurants
WHERE rating >= 4.3
ORDER BY rating DESC
''')

,name,cuisine_type,city,rating
0,Sakura Sushi,Japanese,Springfield,4.7
1,Spice Route,Indian,Capital City,4.6
2,Golden Dragon,Chinese,Springfield,4.5
3,Napoli Pizzeria,Italian,Capital City,4.3


### 5.2 `DISTINCT` — what cuisine types do we have?

In [6]:
run_query("SELECT DISTINCT cuisine_type FROM restaurants")

,cuisine_type
0,Chinese
1,Italian
2,Mexican
3,Japanese
4,American
5,Indian


### 5.3 `MAX` — the most expensive item on each restaurant's menu

In [7]:
run_query('''
SELECT
    r.name AS restaurant,
    MAX(m.price) AS most_expensive_item_price
FROM restaurants r
JOIN menu_items m ON r.restaurant_id = m.restaurant_id
GROUP BY r.name
ORDER BY most_expensive_item_price DESC
''')

,restaurant,most_expensive_item_price
0,Sakura Sushi,16.00
1,Bella Pasta,15.50
2,Golden Dragon,14.99
3,Spice Route,13.99
4,Napoli Pizzeria,13.50
5,The Burger Joint,10.99
6,Taco Fiesta,8.99


### 5.4 `JOIN` — combine restaurants and menu_items

List every menu item along with its restaurant's name and cuisine.

In [8]:
run_query('''
SELECT
    r.name AS restaurant,
    r.cuisine_type,
    m.item_name,
    m.category,
    m.price
FROM restaurants r
JOIN menu_items m ON r.restaurant_id = m.restaurant_id
ORDER BY r.name, m.category
''')

,restaurant,cuisine_type,item_name,category,price
0,Bella Pasta,Italian,Tiramisu,Dessert,7.50
1,Bella Pasta,Italian,Spaghetti Carbonara,Entree,15.50
2,Bella Pasta,Italian,Margherita Pizza,Entree,12.00
3,Golden Dragon,Chinese,Vegetable Spring Rolls,Appetizer,6.99
4,Golden Dragon,Chinese,Kung Pao Chicken,Entree,14.99
5,Golden Dragon,Chinese,Fried Rice,Entree,9.99
6,Napoli Pizzeria,Italian,Caprese Salad,Appetizer,7.00
7,Napoli Pizzeria,Italian,Pepperoni Pizza,Entree,13.50
8,Sakura Sushi,Japanese,Salmon Nigiri (2pc),Appetizer,6.00
9,Sakura Sushi,Japanese,Miso Soup,Appetizer,3.50


### 5.5 Three-way `JOIN` — see who ordered what, from where

Join all three tables together: `orders` -> `menu_items` -> `restaurants`.

In [9]:
run_query('''
SELECT
    o.order_date,
    o.customer_name,
    r.name AS restaurant,
    m.item_name,
    o.quantity,
    m.price,
    ROUND(o.quantity * m.price, 2) AS line_total
FROM orders o
JOIN menu_items m ON o.item_id = m.item_id
JOIN restaurants r ON m.restaurant_id = r.restaurant_id
ORDER BY o.order_date, o.customer_name
''')

,order_date,customer_name,restaurant,item_name,quantity,price,line_total
0,2026-08-01,Alice Chen,Golden Dragon,Kung Pao Chicken,2,14.99,29.98
1,2026-08-01,Alice Chen,Golden Dragon,Fried Rice,1,9.99,9.99
2,2026-08-02,Brian Kim,Bella Pasta,Margherita Pizza,1,12.00,12.00
3,2026-08-02,Brian Kim,Bella Pasta,Tiramisu,2,7.50,15.00
4,2026-08-02,Carla Diaz,Taco Fiesta,Chicken Tacos (3),3,8.99,26.97
5,2026-08-02,Carla Diaz,Taco Fiesta,Churros,2,4.50,9.00
6,2026-08-03,Dan Osei,Sakura Sushi,Dragon Roll,1,16.00,16.00
7,2026-08-03,Dan Osei,Sakura Sushi,Salmon Nigiri (2pc),2,6.00,12.00
8,2026-08-03,Ella Novak,The Burger Joint,Classic Cheeseburger,2,10.99,21.98
9,2026-08-03,Ella Novak,The Burger Joint,Chocolate Milkshake,2,5.50,11.00


### 5.6 `GROUP BY` + `SUM` + `COUNT` — total revenue and orders per restaurant

In [10]:
run_query('''
SELECT
    r.name AS restaurant,
    COUNT(o.order_id) AS num_orders,
    SUM(o.quantity) AS items_sold,
    ROUND(SUM(o.quantity * m.price), 2) AS total_revenue
FROM orders o
JOIN menu_items m ON o.item_id = m.item_id
JOIN restaurants r ON m.restaurant_id = r.restaurant_id
GROUP BY r.name
ORDER BY total_revenue DESC
''')

,restaurant,num_orders,items_sold,total_revenue
0,Bella Pasta,4,7,78.50
1,Napoli Pizzeria,3,5,61.00
2,Golden Dragon,3,4,54.96
3,Spice Route,3,4,45.96
4,Taco Fiesta,3,6,44.96
5,Sakura Sushi,3,4,44.00
6,The Burger Joint,3,5,43.97


### 5.7 `AVG` — average menu price by cuisine type

In [11]:
run_query('''
SELECT
    r.cuisine_type,
    ROUND(AVG(m.price), 2) AS avg_item_price,
    COUNT(m.item_id) AS num_items
FROM restaurants r
JOIN menu_items m ON r.restaurant_id = m.restaurant_id
GROUP BY r.cuisine_type
ORDER BY avg_item_price DESC
''')

,cuisine_type,avg_item_price,num_items
0,Italian,11.10,5
1,Chinese,10.66,3
2,Japanese,8.50,3
3,Indian,7.83,3
4,American,7.16,3
5,Mexican,6.49,3


### 5.8 Most popular menu item overall (by quantity ordered)

In [12]:
run_query('''
SELECT
    m.item_name,
    r.name AS restaurant,
    SUM(o.quantity) AS total_quantity_ordered
FROM orders o
JOIN menu_items m ON o.item_id = m.item_id
JOIN restaurants r ON m.restaurant_id = r.restaurant_id
GROUP BY m.item_name, r.name
ORDER BY total_quantity_ordered DESC
LIMIT 5
''')

,item_name,restaurant,total_quantity_ordered
0,Chicken Tacos (3),Taco Fiesta,4
1,Margherita Pizza,Bella Pasta,4
2,Pepperoni Pizza,Napoli Pizzeria,4
3,Chicken Tikka Masala,Spice Route,3
4,Classic Cheeseburger,The Burger Joint,3


### 5.9 Bonus: vegetarian options per restaurant, only where there are 2 or more (`HAVING`)

In [13]:
run_query('''
SELECT
    r.name AS restaurant,
    COUNT(m.item_id) AS vegetarian_items
FROM restaurants r
JOIN menu_items m ON r.restaurant_id = m.restaurant_id
WHERE m.is_vegetarian = 1
GROUP BY r.name
HAVING COUNT(m.item_id) >= 2
ORDER BY vegetarian_items DESC
''')

,restaurant,vegetarian_items
0,The Burger Joint,2
1,Taco Fiesta,2
2,Spice Route,2
3,Golden Dragon,2
4,Bella Pasta,2


## 6. Close the connection

Always close the connection when you're done.

In [14]:
conn.close()
print("Connection closed.")

Connection closed.
